In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so 'src' is importable
repo_root = str(Path.cwd().parents[1]) if Path.cwd().name == 'research_questions' else str(Path.cwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
from src.analysis.get_results import get_pandas_dataset
from src.analysis.dataset_graphs import plot_graphs_of_dataset_loc
from src.analysis.get_tables_results import create_table_cleaned
from src.analysis.get_results import bar_chart_fix_position_cleaned
from src.analysis.get_results import sucess_vs_position_cleaned,get_latex_table_with_verif_stats
from src.analysis.get_results import compute_stats_tests

import src.config as gl
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
RESULT_DIR = gl.BASE_PATH / "results/dafny_llm_results_rq1__best_overall"
DATASET_DIR = gl.DAFNY_ASSERTION_DATASET
print(DATASET_DIR)
print(RESULT_DIR)
verif_data_pd = get_pandas_dataset(DATASET_DIR, RESULT_DIR)
verif_data_pd  = verif_data_pd.assign(success=lambda d: d['verif_sucess'] > 0) 

In [ ]:
# Graphs of position evaluation
test_models = {
    "claude-haiku-4.5__loc_LAUREL_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "Laurel$_{fl}$",
    "claude-haiku-4.5__loc_LLM_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "LLM$_{fl}$",
    "claude-haiku-4.5__loc_LAUREL_BETTER_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "Laurel$_{fl+}$",
    "claude-haiku-4.5__loc_LLM_EXAMPLE_inf_LLM_EXAMPLE_sExPos_DYNAMIC_nExPos_3_aExPos_0.25_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "LLMEX$_{fl}$",
    "claude-haiku-4.5__loc_HYBRID_inf_LLM_EXAMPLE_sExPos_DYNAMIC_nExPos_3_aExPos_0.25_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "Hybrid$_{fl}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "GrTru$_{fl}$",

    "claude-opus-4.5__loc_LAUREL_BETTER_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "OPUS Laurel$_{fl+}$",
    "claude-opus-4.5__loc_LLM_EXAMPLE_inf_LLM_EXAMPLE_sExPos_DYNAMIC_nExPos_3_aExPos_0.25_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "OPUS LLMEX$_{fl}$",
    "claude-opus-4.5__loc_HYBRID_inf_LLM_EXAMPLE_sExPos_DYNAMIC_nExPos_3_aExPos_0.25_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "OPUS Hybrid$_{fl}$",
    "claude-opus-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "OPUS GrTru$_{fl}$"
}

#verif_data_pd = verif_data_pd[verif_data_pd["benchmark"] == "w/o-1"]
#verif_data_pd = verif_data_pd[verif_data_pd["benchmark"] != "w/o-1"]

dual_name = "rq1__best_overall.pdf"
images_p = gl.BASE_PATH / "images"
info = get_latex_table_with_verif_stats(verif_data_pd, "Verification success rate of each approach across benchmark categories, using the best retrieval strategy MulEmb$0.25_{in}$.", "tbl:assertion-inference-verification", test_models)
print(info)
sucess_vs_position_cleaned(verif_data_pd ,"DOUBLE",   test_models, images_p / dual_name)

In [ ]:
verif_renamed = verif_data_pd.copy()
verif_renamed["llm"] = verif_renamed["llm"].map(test_models)

# OPUS
print("##### Comparing OPUS Laurel$_{fl+}$ with LLMEX$_{fl}$ #####")
compute_stats_tests("OPUS Laurel$_{fl+}$","OPUS LLMEX$_{fl}$", verif_renamed)

print("##### Comparing OPUS Laurel$_{fl+}$ with Hybrid_{fl}$ #####")
compute_stats_tests("OPUS Laurel$_{fl+}$","OPUS Hybrid$_{fl}$", verif_renamed)

print("##### Comparing OPUS Hybrid$_{fl}$ with LLMEX$_{fl}$ #####")
compute_stats_tests("OPUS Hybrid$_{fl}$","OPUS LLMEX$_{fl}$", verif_renamed)

# HAIKU
print("##### Comparing Laurel$_{fl+}$ with LLMEX$_{fl}$ #####")
compute_stats_tests("Laurel$_{fl+}$","LLMEX$_{fl}$", verif_renamed)


print("##### Comparing Laurel$_{fl+}$ with Hybrid_{fl}$ #####")
compute_stats_tests("Laurel$_{fl+}$","Hybrid$_{fl}$", verif_renamed)

print("##### Comparing Hybrid$_{fl}$ with LLMEX$_{fl}$ #####")
compute_stats_tests("Hybrid$_{fl}$","LLMEX$_{fl}$", verif_renamed)
